In [1]:
import os
import urllib.request

if not os.path.exists("the-verdict.txt"):
    url = ("https://raw.githubusercontent.com/rasbt/"
            "LLMs-from-scratch/main/ch02/01_main-chapter-code/"
            "the-verdict.txt")
    file_path = "the-verdict.txt"
    urllib.request.urlretrieve(url, file_path)

In [2]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

In [27]:
#raw_text

In [4]:
import re

result = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
result = [item.strip() for item in result if item.strip()]
preprocessed = result

In [5]:
len(preprocessed)

4690

In [8]:
all_words = sorted(set(preprocessed))
vocabulary_size = len(all_words)
print(vocabulary_size)

1130


In [9]:
vocabulary = {token:integer for integer, token in enumerate(all_words)}

In [11]:
#vocabulary

In [28]:
class SimpleTokenizerV1:
    def __init__(self, vocab):
        self.str_to_int = vocabulary
        self.int_to_str = {i:s for s, i in vocabulary.items()}

    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text) 
        preprocessed = [item.strip() for item in preprocessed if item.strip()]
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids

    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        #Replace spaces before the specified punctuation
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        return text
        

In [35]:
class SimpleTokenizerV2:
    def __init__(self, vocab):
        self.str_to_int = vocabulary
        self.int_to_str = {i:s for s, i in vocabulary.items()}

    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text) 
        preprocessed = [item.strip() for item in preprocessed if item.strip()]
        preprocessed = [
            item if item in self.str_to_int
            else "<|unk|>" for item in preprocessed
        ]
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids

    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        #Replace spaces before the specified punctuation
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        return text
        

In [36]:
tokeniser = SimpleTokenizerV2(vocabulary)

In [37]:
text = "Hello, do you like tea. is this-- a test?"

In [38]:
all_tokens = sorted(list(set(preprocessed)))
all_tokens.extend(["<|endoftext|>", "<|unk|>"])

vocabulary = {token:integer for integer, token in enumerate(all_tokens)}

In [39]:
for i, item in enumerate(list(vocabulary.items())[-5:]):
    print(item)

('younger', 1127)
('your', 1128)
('yourself', 1129)
('<|endoftext|>', 1130)
('<|unk|>', 1131)


In [40]:
tokeniser = SimpleTokenizerV2(vocabulary)

In [41]:
tokeniser.encode(text)

[1131, 5, 355, 1126, 628, 975, 7, 584, 999, 6, 115, 1131, 10]

In [42]:
tokeniser.decode(tokeniser.encode(text))

'<|unk|>, do you like tea. is this -- a <|unk|>?'

In [43]:
import tiktoken

ModuleNotFoundError: No module named 'tiktoken'

In [44]:
pip install tiktoken

   ---------------------------------------- 0.0/918.7 kB ? eta -:--:--
   ---------------------------------------- 0.0/918.7 kB ? eta -:--:--
   ---------------------------------------- 0.0/918.7 kB ? eta -:--:--
   ---------------------------------------- 0.0/918.7 kB ? eta -:--:--
   ---------------------------------------- 0.0/918.7 kB ? eta -:--:--
   ---------------------------------------- 0.0/918.7 kB ? eta -:--:--
   ---------------------------------------- 0.0/918.7 kB ? eta -:--:--
   ----------- ---------------------------- 262.1/918.7 kB ? eta -:--:--
   --------------------- ---------------- 524.3/918.7 kB 839.1 kB/s eta 0:00:01
   --------------------- ---------------- 524.3/918.7 kB 839.1 kB/s eta 0:00:01
   -------------------------------- ----- 786.4/918.7 kB 807.1 kB/s eta 0:00:01
   ---------------------------------------- 918.7/918.7 kB 784.5 kB/s  0:00:02

   -------------------- ------------------- 1/2 [tiktoken]
   ---------------------------------------- 2/2 [ti


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: C:\Users\kenne\AppData\Local\Programs\Python\Python314\python.exe -m pip install --upgrade pip


2.5 Byte Code Pairing

In [47]:
import tiktoken

In [48]:
tiktoken.__version__

'0.13.0'

In [49]:
tokeniser = tiktoken.get_encoding("gpt2")

In [50]:
tokeniser.encode("Hello world")

[15496, 995]

In [51]:
tokeniser.decode(tokeniser.encode("Hello world"))

'Hello world'

In [52]:
text = (
    "Hello, do you like tea? <|endoftext|> In the sunlit terraces"
    "of someunknownPlace."
)
tokeniser.encode(text, allowed_special={"<|endoftext|>"})

[15496,
 11,
 466,
 345,
 588,
 8887,
 30,
 220,
 50256,
 554,
 262,
 4252,
 18250,
 8812,
 2114,
 1659,
 617,
 34680,
 27271,
 13]

Data Sampling with a sliding window

In [53]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

enc_text = tokeniser.encode(raw_text)
print(len(enc_text))

5145


In [54]:
enc_sample = enc_text[50:]

In [55]:
context_size = 4

x = enc_sample[:context_size]
y = enc_sample[1:context_size+1]

print(f"x: {x}")
print(f"y:     {y}") 

x: [290, 4920, 2241, 287]
y:     [4920, 2241, 287, 257]


In [58]:
for i in range(1, context_size + 1):
    context = enc_sample[:i]
    desired = enc_sample[i]

    print(tokeniser.decode(context), "--------->", tokeniser.decode([desired]))

 and --------->  established
 and established --------->  himself
 and established himself --------->  in
 and established himself in --------->  a


In [59]:
import torch

ModuleNotFoundError: No module named 'torch'

In [60]:
pip install torch

   ---------------------------------------- 0.0/122.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/122.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/122.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/122.1 MB ? eta -:--:--
   ---------------------------------------- 0.3/122.1 MB ? eta -:--:--
   ---------------------------------------- 0.3/122.1 MB ? eta -:--:--
   ---------------------------------------- 0.5/122.1 MB 875.8 kB/s eta 0:02:19
   ---------------------------------------- 0.8/122.1 MB 906.7 kB/s eta 0:02:14
   ---------------------------------------- 0.8/122.1 MB 906.7 kB/s eta 0:02:14
   ---------------------------------------- 1.0/122.1 MB 852.2 kB/s eta 0:02:22
   ---------------------------------------- 1.0/122.1 MB 852.2 kB/s eta 0:02:22
   ---------------------------------------- 1.3/122.1 MB 733.5 kB/s eta 0:02:45
   ---------------------------------------- 1.3/122.1 MB 733.5 kB/s eta 0:02:45
   -----------


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: C:\Users\kenne\AppData\Local\Programs\Python\Python314\python.exe -m pip install --upgrade pip


In [61]:
torch.__version__

NameError: name 'torch' is not defined

In [62]:
import torch

In [63]:
torch.__version__

'2.13.0+cpu'

In [67]:
from torch.utils.data import Dataset, DataLoader

class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokeniser, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        #Tokenise the entire text
        token_ids = tokeniser.encode(txt, allowed_special={"<|endoftext|>"})
        assert len(token_ids) > max_length, "Number of tokenized inputs must at least be equal to max_length+1"
    
        #Use a sliding window to chunk the book into overlapping sequences of max_length
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1: i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]
        

In [68]:
def create_dataloader_v1(txt, batch_size=4, max_length=256, stride=128, shuffle=True, drop_last=True, num_workers=0):
    
    #Initialise the tokeniser
    tokeniser = tiktoken.get_encoding("gpt2")
    
    #Create dataset
    dataset = GPTDatasetV1(txt, tokeniser, max_length, stride)

    #Create dataloader
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, drop_last=drop_last, num_workers=num_workers)

    return dataloader

In [69]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

In [71]:
dataloader = create_dataloader_v1(raw_text, batch_size=1, max_length=4, stride=1, shuffle=False)
data_iter = iter(dataloader)
first_batch = next(data_iter)
print(first_batch)

[tensor([[  40,  367, 2885, 1464]]), tensor([[ 367, 2885, 1464, 1807]])]


In [72]:
second_batch = next(data_iter)
print(second_batch)

[tensor([[ 367, 2885, 1464, 1807]]), tensor([[2885, 1464, 1807, 3619]])]


In [73]:
dataloader = create_dataloader_v1(raw_text, batch_size=8, max_length=4, stride=4, shuffle=False)
data_iter = iter(dataloader)
inputs, targets = next(data_iter)
print("Inputs:\n", inputs)
print("\nTargets:\n:", targets)

Inputs:
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])

Targets:
: tensor([[  367,  2885,  1464,  1807],
        [ 3619,   402,   271, 10899],
        [ 2138,   257,  7026, 15632],
        [  438,  2016,   257,   922],
        [ 5891,  1576,   438,   568],
        [  340,   373,   645,  1049],
        [ 5975,   284,   502,   284],
        [ 3285,   326,    11,   287]])
